In [1]:
import urllib3
import requests

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

api_url = "https://115.159.159.82/api/devices"

r = requests.get(
    api_url,
    params={"page": 1, "page_size": 50},
    verify=False,
    timeout=10
)

print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))

data = r.json()

print()
print("顶层字段:", list(data.keys()))
print("page:", data["page"])
print("page_size:", data["page_size"])
print("total:", data["total"])
print("本页 items 条数:", len(data["items"]))
print()
print("第一条设备记录:")
print(data["items"][0])


status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json

顶层字段: ['page', 'page_size', 'total', 'items']
page: 1
page_size: 50
total: 5000
本页 items 条数: 50

第一条设备记录:
{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}


In [2]:
total = data["total"]
page_size = data["page_size"]

# 向上取整：5000 / 50 = 100，不多不少；但如果 5001 条就要 101 页
total_pages = (total + page_size - 1) // page_size

print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)


total: 5000
page_size: 50
total_pages: 100


In [3]:
devices = []

for page in range(1, total_pages + 1):
    r = requests.get(
        api_url,
        params={"page": page, "page_size": page_size},
        verify=False,
        timeout=10
    )
    r.raise_for_status()
    
    page_data = r.json()
    devices.extend(page_data["items"])

print("采集到的设备数:", len(devices))
print("唯一 device_id 数:", len(set(x["device_id"] for x in devices)))
print()
print("第一条:", devices[0])


采集到的设备数: 5000
唯一 device_id 数: 5000

第一条: {'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}


In [4]:
from bs4 import BeautifulSoup

base_url = "https://115.159.159.82"
current_url = base_url + "/companies"

r = requests.get(current_url, verify=False, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

cards = soup.select("article.company-card")

print("本页企业数:", len(cards))
print()
print("第一条卡片的 HTML:")
print(cards[0].prettify()[:1200])


本页企业数: 20

第一条卡片的 HTML:
<article class="company-card" data-company-id="C000001">
 <div class="company-card-top">
  <span class="mono">
   C000001
  </span>
  <span class="badge active">
   运营中
  </span>
 </div>
 <h2>
  <a href="/companies/C000001">
   临港精工001有限公司
  </a>
 </h2>
 <dl>
  <div>
   <dt>
    所属行业
   </dt>
   <dd>
    汽车零部件
   </dd>
  </div>
  <div>
   <dt>
    所在区域
   </dt>
   <dd>
    智造一区
   </dd>
  </div>
  <div>
   <dt>
    员工规模
   </dt>
   <dd>
    88 人
   </dd>
  </div>
  <div>
   <dt>
    设备数量
   </dt>
   <dd>
    6 台
   </dd>
  </div>
 </dl>
 <a class="detail-link" href="/companies/C000001">
  查看企业档案
  <span aria-hidden="true">
   →
  </span>
 </a>
</article>



In [5]:
card = cards[0]

company_id = card.get("data-company-id")
name = card.select_one("h2 a").get_text(strip=True)
status = card.select_one(".badge").get_text(strip=True)

details = {}
for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

print("details 字典:", details)

industry = details["所属行业"]
zone = details["所在区域"]
device_count = int(details["设备数量"].replace(" 台", ""))

print()
print("company_id:", company_id)
print("name:", name)
print("industry:", industry)
print("zone:", zone)
print("device_count:", device_count)
print("status:", status)


details 字典: {'所属行业': '汽车零部件', '所在区域': '智造一区', '员工规模': '88 人', '设备数量': '6 台'}

company_id: C000001
name: 临港精工001有限公司
industry: 汽车零部件
zone: 智造一区
device_count: 6
status: 运营中


In [6]:
for u in ["https://115.159.159.82/companies",
          "https://115.159.159.82/companies?page=2"]:
    r = requests.get(u, verify=False, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")
    nav = soup.select_one("nav.pagination")
    
    print("=== " + u)
    print("分页栏 HTML:", nav)
    print("里面有", len(nav.select("a")), "个 <a>")
    print("select_one('nav.pagination a') 拿到的是：",
          soup.select_one("nav.pagination a").get_text(strip=True))
    print()



=== https://115.159.159.82/companies
分页栏 HTML: <nav aria-label="企业分页" class="pagination"><span>上一页</span><b>1</b><a href="/companies?page=2">下一页</a></nav>
里面有 1 个 <a>
select_one('nav.pagination a') 拿到的是： 下一页

=== https://115.159.159.82/companies?page=2
分页栏 HTML: <nav aria-label="企业分页" class="pagination"><a href="/companies?page=1">上一页</a><b>2</b><a href="/companies?page=3">下一页</a></nav>
里面有 2 个 <a>
select_one('nav.pagination a') 拿到的是： 上一页



In [7]:
r = requests.get("https://115.159.159.82/companies?page=2", verify=False, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

for sel in ["nav.pagination a",
            "nav.pagination a:last-of-type",
            "nav.pagination a:last-child"]:
    el = soup.select_one(sel)
    print(f"{sel:35s} -> {el.get_text(strip=True) if el else None}  href={el.get('href') if el else None}")


nav.pagination a                    -> 上一页  href=/companies?page=1
nav.pagination a:last-of-type       -> 下一页  href=/companies?page=3
nav.pagination a:last-child         -> 下一页  href=/companies?page=3


In [8]:
from urllib.parse import urljoin

base_url = "https://115.159.159.82"
current_url = base_url + "/companies"

companies, page_counts, page_no = [], [], 0

while current_url and page_no < 3:
    r = requests.get(current_url, verify=False, timeout=10)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    cards = soup.select("article.company-card")
    page_no += 1
    page_counts.append(len(cards))

    for card in cards:
        details = {}
        for item in card.select("dl > div"):
            details[item.select_one("dt").get_text(strip=True)] = \
                item.select_one("dd").get_text(strip=True)
        companies.append({
            "company_id": card.get("data-company-id"),
            "name": card.select_one("h2 a").get_text(strip=True),
            "industry": details["所属行业"],
            "zone": details["所在区域"],
            "device_count": int(details["设备数量"].replace(" 台", "")),
            "status": card.select_one(".badge").get_text(strip=True),
        })

    next_link = None
    for a in soup.select("nav.pagination a"):        # 关键：按文本找"下一页"
        if a.get_text(strip=True) == "下一页":
            next_link = a
            break
    current_url = urljoin(current_url, next_link["href"]) if next_link else None

print("每页企业数:", page_counts)
print("采集企业总数:", len(companies))
print("唯一 company_id 数:", len(set(c["company_id"] for c in companies)))


每页企业数: [20, 20, 20]
采集企业总数: 60
唯一 company_id 数: 60


In [9]:
from collections import Counter

device_counts = Counter(x["company_id"] for x in devices)

mismatch_count = 0
for company in companies:
    cid = company["company_id"]
    expected = company["device_count"]
    actual = device_counts.get(cid, 0)
    if expected != actual:
        mismatch_count += 1

print("Company 总数:", len(companies))
print("唯一 company_id 数:", len(set(c["company_id"] for c in companies)))
print("Device 总数:", len(devices))
print("唯一 device_id 数:", len(set(d["device_id"] for d in devices)))
print("Company–Device 不一致企业数:", mismatch_count)


Company 总数: 60
唯一 company_id 数: 60
Device 总数: 5000
唯一 device_id 数: 5000
Company–Device 不一致企业数: 0


In [10]:
import json, os
os.makedirs("data", exist_ok=True)
with open("data/companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)
with open("data/devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)
print("已保存")

已保存
